# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

According to the UCI Bank Marketing Dataset documentation, this dataset represents multiple direct marketing campaigns carried out by a Portuguese banking institution. Specifically, it comprises data from phone‑based telemarketing campaigns conducted between May 2008 and November 2010, during which more than one contact per client was often required to determine if they would subscribe to a term deposit product.

However, the repository itself doesn’t explicitly list a single numeric count of distinct “marketing campaigns” as separate events—but the dataset covers the entire period of campaign activity over those years and includes variables (like campaign, pdays, previous, and poutcome) that reflect the sequence and count of contacts within and across campaigns.

The dataset represents multiple telemarketing campaigns carried out by a Portuguese bank between May 2008 and November 2010. It contains 41,188 client-contact records, with an average of 2.57 contacts per client per campaign. A total of 5625 clients had been contacted previously, and some clients required up to 56 contacts in a single campaign, highlighting the persistence needed in telemarketing

**Business Problem Statement**

A Portuguese banking institution conducted multiple telemarketing campaigns between 2008 and 2010 to promote term deposit subscriptions. These campaigns involved contacting clients by phone, often multiple times, to determine whether they would subscribe to a term deposit product.

The core business challenge is that only a small percentage of clients subscribe, while the majority do not. Since telemarketing campaigns require significant time, human effort, and operational cost, contacting every client is inefficient and expensive.

The objective of this project is to develop predictive classification models that can identify clients who are most likely to subscribe to a term deposit using available demographic and financial information. By accurately predicting potential subscribers:

The bank can optimize marketing resources

Reduce unnecessary client contacts

Increase campaign conversion rates

Improve return on investment (ROI)

The ultimate goal is to support data-driven decision-making in future marketing campaigns by identifying high-probability customers before outreach efforts begin.

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('bank-additional-full.csv', sep = ';')

In [ ]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [ ]:
# Number of clients with previous contacts
print("Clients contacted previously:", sum(df['previous'] > 0))

# Average number of contacts per campaign
print("Average contacts per campaign:", df['campaign'].mean())

# Maximum number of contacts in a single campaign
print("Max contacts in a campaign:", df['campaign'].max())

Clients contacted previously: 5625
Average contacts per campaign: 2.567592502670681
Max contacts in a campaign: 56


In [ ]:
# Count of clients with at least one contact in a campaign
unique_campaigns = df[df['campaign'] > 0].shape[0]
print("Number of campaign entries (rows with contacts):", unique_campaigns)

Number of campaign entries (rows with contacts): 41188


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [ ]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [ ]:
# Check for actual missing values (NaN)
print(df.isnull().sum())

age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64


No missing values because the dataset uses "unknown" instead of NaN.

In [ ]:
# Count unknowns in each categorical feature
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

for col in categorical_cols:
    print(f"{col} value counts:\n{df[col].value_counts()}\n")

job value counts:
job
admin.           10422
blue-collar       9254
technician        6743
services          3969
management        2924
retired           1720
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
unknown            330
Name: count, dtype: int64

marital value counts:
marital
married     24928
single      11568
divorced     4612
unknown        80
Name: count, dtype: int64

education value counts:
education
university.degree      12168
high.school             9515
basic.9y                6045
professional.course     5243
basic.4y                4176
basic.6y                2292
unknown                 1731
illiterate                18
Name: count, dtype: int64

default value counts:
default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64

housing value counts:
housing
yes        21576
no         18622
unknown      990
Name: count, dtype: int64

loan value counts:
loan
no         33

Unknown Value imputed with Mode of that feature

In [ ]:
# List of categorical columns where "unknown" appears
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

# Impute "unknown" with mode for each column
for col in categorical_cols:
    mode_value = df[col].mode()[0]  # get the most frequent value
    df[col] = df[col].replace('unknown', mode_value)

# Verify the replacement
for col in categorical_cols:
    print(f"{col} value counts after imputation:\n{df[col].value_counts()}\n")

job value counts after imputation:
job
admin.           10752
blue-collar       9254
technician        6743
services          3969
management        2924
retired           1720
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
Name: count, dtype: int64

marital value counts after imputation:
marital
married     25008
single      11568
divorced     4612
Name: count, dtype: int64

education value counts after imputation:
education
university.degree      13899
high.school             9515
basic.9y                6045
professional.course     5243
basic.4y                4176
basic.6y                2292
illiterate                18
Name: count, dtype: int64

default value counts after imputation:
default
no     41185
yes        3
Name: count, dtype: int64

housing value counts after imputation:
housing
yes    22566
no     18622
Name: count, dtype: int64

loan value counts after imputation:
loan
no     34940
yes     6248
Name:

In [ ]:
# Check data types
print(df.dtypes)

age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                  object
dtype: object


### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

**Business Objective**

The goal of this analysis is to predict whether a bank client will subscribe to a term deposit based on demographic, financial, and campaign-related information. By building predictive models using features such as age, job, marital status, account balance, previous campaign interactions, and contact details, the bank aims to:

Identify clients most likely to subscribe, improving the efficiency of future marketing campaigns.

Reduce marketing costs by targeting only the most promising prospects.

Provide actionable insights into which client attributes and campaign strategies are most effective for increasing term deposit subscriptions.

**Outcome:** The predictive model will help the bank make data-driven decisions to maximize campaign success while minimizing unnecessary client contacts.

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [ ]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

Step 1: Select Features and Target

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Bank client features (columns 1–8)
bank_features = ['age', 'job', 'marital', 'education', 'default','housing', 'loan']

# Target
target = 'y'

X = df[bank_features].copy()
y = df[target].copy()

Step 2: Encode Categorical Features Using LabelEncoder

In [ ]:
# Initialize LabelEncoder
le = LabelEncoder()

# Identify categorical columns in bank features
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan']

# Apply LabelEncoder to categorical columns
for col in categorical_cols:
    X[col] = le.fit_transform(X[col])

# Encode target column
y = le.fit_transform(y)  # 'yes' -> 1, 'no' -> 0

# Check the first few rows after encoding
X.head()

,age,job,marital,education,default,housing,loan
0,56,3,1,0,0,0,0
1,57,7,1,3,0,0,0
2,37,7,1,3,0,1,0
3,40,0,1,1,0,0,0
4,56,7,1,3,0,0,1


### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [ ]:
# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Training set size: (32950, 7)
Test set size: (8238, 7)


### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

Step 1: Check Class Distribution

In [ ]:
# Count of each class
import numpy as np

unique, counts = np.unique(y, return_counts=True)
class_counts = dict(zip(unique, counts))
print("Class distribution:", class_counts)

# Percentage of majority class
majority_class_percentage = max(counts) / sum(counts)
print("Baseline accuracy (majority class):", majority_class_percentage)

Class distribution: {np.int64(0): np.int64(36548), np.int64(1): np.int64(4640)}
Baseline accuracy (majority class): 0.8873458288821987


**Baseline Performance**

Class distribution:

0 (No subscription) → 36,548 clients

1 (Yes subscription) → 4,640 clients

Baseline accuracy (majority class): ~88.7%

### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# Fit the model on training data
lr_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
# Predict on test data
y_pred = lr_model.predict(X_test)

### Problem 9: Score the Model

What is the accuracy of your model?

In [ ]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Logistic Regression Accuracy:", accuracy)

# Classification report (precision, recall, F1-score)
print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)

Logistic Regression Accuracy: 0.8873512988589464

Classification Report:
               precision    recall  f1-score   support

           0       0.89      1.00      0.94      7310
           1       0.00      0.00      0.00       928

    accuracy                           0.89      8238
   macro avg       0.44      0.50      0.47      8238
weighted avg       0.79      0.89      0.83      8238


Confusion Matrix:
 [[7310    0]
 [ 928    0]]


### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [ ]:
import time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import pandas as pd

# Dictionary to store results
results = {
    'Model': [],
    'Train Time (s)': [],
    'Train Accuracy': [],
    'Test Accuracy': []
}

# List of models to evaluate
models = [
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')),
    ('K-Nearest Neighbors', KNeighborsClassifier()),
    ('Decision Tree', DecisionTreeClassifier(random_state=42)),
    ('Support Vector Machine', SVC())
]

# Fit, predict, and record results
for name, model in models:
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time

    # Predict on train and test
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Record results
    results['Model'].append(name)
    results['Train Time (s)'].append(round(train_time, 3))
    results['Train Accuracy'].append(round(accuracy_score(y_train, y_train_pred), 3))
    results['Test Accuracy'].append(round(accuracy_score(y_test, y_test_pred), 3))

# Convert results to DataFrame
results_df = pd.DataFrame(results)
results_df

,Model,Train Time (s),Train Accuracy,Test Accuracy
0,Logistic Regression,0.520,0.558,0.555
1,K-Nearest Neighbors,0.159,0.890,0.878
2,Decision Tree,0.103,0.908,0.872
3,Support Vector Machine,17.663,0.887,0.887


### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score, classification_report
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Dictionary to store results
tuned_results = {
    'Model': [],
    'Best Parameters': [],
    'Train Accuracy': [],
    'Test Accuracy': [],
    'F1-Score (Test)': []
}

# 1️⃣ Logistic Regression
lr_params = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['lbfgs', 'liblinear'],
    'class_weight': ['balanced']
}

lr_grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42),
                       param_grid=lr_params,
                       scoring='f1',
                       cv=5,
                       n_jobs=-1)
lr_grid.fit(X_train, y_train)

y_train_pred = lr_grid.best_estimator_.predict(X_train)
y_test_pred = lr_grid.best_estimator_.predict(X_test)

tuned_results['Model'].append('Logistic Regression')
tuned_results['Best Parameters'].append(lr_grid.best_params_)
tuned_results['Train Accuracy'].append(round(accuracy_score(y_train, y_train_pred), 3))
tuned_results['Test Accuracy'].append(round(accuracy_score(y_test, y_test_pred), 3))
tuned_results['F1-Score (Test)'].append(round(f1_score(y_test, y_test_pred), 3))

print("Logistic Regression Classification Report:\n", classification_report(y_test, y_test_pred, zero_division=0))

# 2️⃣ K-Nearest Neighbors
knn_params = {
    'n_neighbors': [3,5,7],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_grid = GridSearchCV(KNeighborsClassifier(),
                        param_grid=knn_params,
                        scoring='f1',
                        cv=5,
                        n_jobs=-1)
knn_grid.fit(X_train, y_train)

y_train_pred = knn_grid.best_estimator_.predict(X_train)
y_test_pred = knn_grid.best_estimator_.predict(X_test)

tuned_results['Model'].append('K-Nearest Neighbors')
tuned_results['Best Parameters'].append(knn_grid.best_params_)
tuned_results['Train Accuracy'].append(round(accuracy_score(y_train, y_train_pred), 3))
tuned_results['Test Accuracy'].append(round(accuracy_score(y_test, y_test_pred), 3))
tuned_results['F1-Score (Test)'].append(round(f1_score(y_test, y_test_pred), 3))

print("KNN Classification Report:\n", classification_report(y_test, y_test_pred, zero_division=0))

# 3️⃣ Decision Tree
dt_params = {
    'max_depth': [3,5,7,10,None],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,2,5],
    'class_weight': ['balanced']
}

dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42),
                       param_grid=dt_params,
                       scoring='f1',
                       cv=5,
                       n_jobs=-1)
dt_grid.fit(X_train, y_train)

y_train_pred = dt_grid.best_estimator_.predict(X_train)
y_test_pred = dt_grid.best_estimator_.predict(X_test)

tuned_results['Model'].append('Decision Tree')
tuned_results['Best Parameters'].append(dt_grid.best_params_)
tuned_results['Train Accuracy'].append(round(accuracy_score(y_train, y_train_pred), 3))
tuned_results['Test Accuracy'].append(round(accuracy_score(y_test, y_test_pred), 3))
tuned_results['F1-Score (Test)'].append(round(f1_score(y_test, y_test_pred), 3))

print("Decision Tree Classification Report:\n", classification_report(y_test, y_test_pred, zero_division=0))

# 4️⃣ Support Vector Machine
svm_params = {
    'C': [0.1, 1],
    'kernel': ['linear'],
    'class_weight': ['balanced']
}
X_train_sample, _, y_train_sample, _ = train_test_split(X_train, y_train, train_size=0.2, stratify=y_train, random_state=42)
svm_grid = GridSearchCV(SVC(),
                        param_grid=svm_params,
                        scoring='f1',
                        cv=3,
                        n_jobs=-1)
svm_grid.fit(X_train_sample, y_train_sample)

y_train_pred = svm_grid.best_estimator_.predict(X_train)
y_test_pred = svm_grid.best_estimator_.predict(X_test)

tuned_results['Model'].append('Support Vector Machine')
tuned_results['Best Parameters'].append(svm_grid.best_params_)
tuned_results['Train Accuracy'].append(round(accuracy_score(y_train, y_train_pred), 3))
tuned_results['Test Accuracy'].append(round(accuracy_score(y_test, y_test_pred), 3))
tuned_results['F1-Score (Test)'].append(round(f1_score(y_test, y_test_pred), 3))

print("SVM Classification Report:\n", classification_report(y_test, y_test_pred, zero_division=0))

# Convert results to DataFrame
tuned_results_df = pd.DataFrame(tuned_results)
tuned_results_df

Logistic Regression Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.52      0.66      7310
           1       0.14      0.60      0.22       928

    accuracy                           0.53      8238
   macro avg       0.52      0.56      0.44      8238
weighted avg       0.82      0.53      0.61      8238

KNN Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.96      0.93      7310
           1       0.24      0.09      0.13       928

    accuracy                           0.86      8238
   macro avg       0.57      0.53      0.53      8238
weighted avg       0.82      0.86      0.84      8238

Decision Tree Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.73      0.81      7310
           1       0.18      0.46      0.26       928

    accuracy                           0.70      8238
   macro avg      

,Model,Best Parameters,Train Accuracy,Test Accuracy,F1-Score (Test)
0,Logistic Regression,"{'C': 0.01, 'class_weight': 'balanced', 'solve...",0.532,0.527,0.223
1,K-Nearest Neighbors,"{'metric': 'manhattan', 'n_neighbors': 3, 'wei...",0.900,0.865,0.132
2,Decision Tree,"{'class_weight': 'balanced', 'max_depth': 5, '...",0.702,0.699,0.256
3,Support Vector Machine,"{'C': 0.1, 'class_weight': 'balanced', 'kernel...",0.534,0.532,0.222
